# Nudge Problem: Nudge vs a smart email workflow

**Question:** the industry rule ("every cart abandoner gets 10%") is a weak opponent. A smarter marketer would (a) wait: send a plain reminder first and give the 10% only if the shopper has still not bought after 24 or 48 hours, or (b) write a segment rule (for example "no discount for returning shoppers"). Does Nudge still win against those? Everything is chosen on **validation** and reported on **test** (Week 3 s22), as in `05_evaluation`.

1. Load the visits and the saved Transformer scores (3 seeds averaged), check the rows line up.
2. Policies, cart abandoners only, one discount per shopper per week (`nudge/evaluate.py` `money`):
   rule (everyone); escalation after D = 24 h and 48 h; the best 2 to 3 condition segment rule found on validation at about the same number of discounts as Nudge; Nudge at 60%; Nudge + escalation (Nudge's pick, sent only if no purchase within 24 h).
3. Compare € wasted, discounts sent, left without a nudge, € wasted per discount, with 95% ranges from resampling shoppers (200 draws).
4. The honest catches, below the table.

**Input:** `data/raw/visits_25pct.parquet`, `results/tuned/transformer_4eb04ce3_s{42,43,44}_{val,test}.npy`, `results/tuning/stage2.csv`.
**Output:** `results/smart_workflows.csv` (and `smart_workflows_segment_search.csv`: the top segment rules on validation).

**Owner:** TBD (Comparison role)

## 0. Imports and settings
`D_HOURS` are the waiting times tried. The score cut-off for Nudge at 60% is the 60th percentile of the **validation** cart abandoners' scores, exactly as in `05_evaluation` section 6.

In [1]:
import sys, pathlib, json, itertools
sys.path.insert(0, str(pathlib.Path.cwd().parent))   # repo root

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score
from IPython.display import display

from nudge.train import SEED
from nudge.evaluate import money, bootstrap

DATA = pathlib.Path("../data/raw/visits_25pct.parquet")
TUNED = pathlib.Path("../results/tuned")
OUT = pathlib.Path("../results")
TAG = "transformer_4eb04ce3"
D_HOURS = (24, 48)
SHARE = 0.60
BOOT_N = 200

## 1. Load visits and Transformer scores
Same row order as `scripts/tune.py` and `05_evaluation` (split column, reset index). The check compares the recomputed validation PR-AUC of each seed with the one `tune.py` wrote.

In [2]:
visits = pd.read_parquet(DATA)
val = visits[visits["split"] == "valid"].reset_index(drop=True)
test = visits[visits["split"] == "test"].reset_index(drop=True)
stage2 = pd.read_csv("../results/tuning/stage2.csv")
S = {"val": [], "test": []}
for seed in (42, 43, 44):
    v, t = np.load(TUNED / f"{TAG}_s{seed}_val.npy"), np.load(TUNED / f"{TAG}_s{seed}_test.npy")
    assert len(v) == len(val) and len(t) == len(test)
    ref = stage2.loc[stage2["tag"] == f"{TAG}_s{seed}", "val_pr_auc"].iloc[0]
    assert abs(average_precision_score(val["label"], v) - ref) < 1e-4, "rows out of order"
    S["val"].append(v); S["test"].append(t)
val["score"], test["score"] = np.mean(S["val"], axis=0), np.mean(S["test"], axis=0)
# Cart abandoners only: that is who every policy is about (money() ignores the rest anyway)
val, test = [d[d["cart_value"] > 0].reset_index(drop=True) for d in (val, test)]
for d in (val, test):
    d["hours_to_buy"] = d["days_to_buy"] * 24      # NaN when the shopper did not come back within 7 days
    d["returning"] = d["tokens"].str[0].eq("returning")
print(f"validation cart abandoners {len(val):,} ({val.label.mean():.1%} come back) | test {len(test):,} ({test.label.mean():.1%})")
print("share of come-backs that happen within 24 h / 48 h (test):",
      f"{(test.hours_to_buy < 24).sum() / test.label.sum():.1%} / {(test.hours_to_buy < 48).sum() / test.label.sum():.1%}")

validation cart abandoners 22,468 (20.9% come back) | test 22,384 (21.8%)
share of come-backs that happen within 24 h / 48 h (test): 41.9% / 66.9%


## 2. The policies as yes/no per visit
- **Rule:** discount everyone.
- **Escalation, D hours:** every cart abandoner gets a reminder at the decision moment; the 10% goes out at hour D to those who have **not** bought by then. So a shopper who bought within D hours gets no discount (the flow saves that money), a shopper who bought later or never gets it. The data holds the real time of the next purchase (`days_to_buy`), so nothing is assumed. The one-per-week rule is applied on the decision time (a small simplification: the real discount goes out D hours later).
- **Nudge at 60%:** discount cart abandoners whose score is under the 60th percentile of the validation cart abandoners.
- **Nudge + escalation:** the same pick, but sent only if no purchase within 24 h.
- **Segment rule:** the best conjunction of 2 to 3 simple conditions (below), chosen on validation.

In [3]:
cut60 = np.quantile(val["score"], SHARE)
print(f"Nudge@60% cut-off (validation 60th percentile of cart abandoners' scores): {cut60:.4f}")

def not_bought_within(d, hours):
    """True where the shopper has NOT bought within `hours` of the decision moment (never = True)."""
    return ~(d["hours_to_buy"] < hours).to_numpy()

def policy_masks(d):
    m = {"Industry rule (everyone gets 10%)": np.ones(len(d), dtype=bool)}
    for h in D_HOURS:
        m[f"Escalation: reminder, 10% after {h} h if not bought"] = not_bought_within(d, h)
    nudge = (d["score"].to_numpy() < cut60)
    m["Nudge@60%"] = nudge
    m["Nudge@60% + escalation 24 h"] = nudge & not_bought_within(d, 24)
    return m

def n_given(d, give):
    return int(money(d, give)["discounts_sent"])
for k, g in policy_masks(val).items():
    print(f"validation | {k:55s} discounts {n_given(val, g):,}")

Nudge@60% cut-off (validation 60th percentile of cart abandoners' scores): 0.2146
validation | Industry rule (everyone gets 10%)                       discounts 16,503
validation | Escalation: reminder, 10% after 24 h if not bought      discounts 15,474
validation | Escalation: reminder, 10% after 48 h if not bought      discounts 14,832
validation | Nudge@60%                                               discounts 12,195
validation | Nudge@60% + escalation 24 h                             discounts 11,503


## 3. Search for the best segment rule on validation
Building blocks (all from what a shop knows at the decision moment): new vs returning, bought in the last 14 days, number of visits in 14 days, cart value above or below a threshold (cart-value percentiles from validation), number of clicks. Every conjunction of 2 and 3 blocks is tried. Keep rules whose number of discounts on validation is within 15% of Nudge@60%'s, then take the one with the lowest € wasted. Test is not touched here.

In [4]:
cv = val["cart_value"]
atoms = {"new visitor": val["returning"].eq(False), "returning visitor": val["returning"],
         "bought in last 14 d": val["bought_14d"], "did not buy in last 14 d": ~val["bought_14d"],
         "no visit in last 14 d": val["visits_14d"].eq(0), "1+ visit in last 14 d": val["visits_14d"].ge(1),
         "3+ visits in last 14 d": val["visits_14d"].ge(3),
         "1 to 3 clicks": val["n_clicks"].le(3), "4+ clicks": val["n_clicks"].ge(4), "8+ clicks": val["n_clicks"].ge(8), "15+ clicks": val["n_clicks"].ge(15), "2 or fewer clicks": val["n_clicks"].le(2)}
for q in (0.1, 0.25, 0.5, 0.75, 0.9):
    t = cv.quantile(q)
    atoms[f"cart value <= €{t:,.0f}"] = cv.le(t)
    atoms[f"cart value > €{t:,.0f}"] = cv.gt(t)
atoms = {k: v.to_numpy() for k, v in atoms.items()}
target = n_given(val, np.asarray(policy_masks(val)["Nudge@60%"]))
rows = []
for size in (2, 3):
    for combo in itertools.combinations(atoms, size):
        kinds = [c.split(" ")[0] + (" <=" if "<=" in c else " >" if ">" in c else "") for c in combo if c.startswith("cart value")]
        if len(kinds) != len(set(kinds)) or ("returning visitor" in combo and "new visitor" in combo):   # no redundant pairs
            continue
        give = np.logical_and.reduce([atoms[c] for c in combo])
        if not 0.5 * target <= give.sum() <= 2 * target:      # cheap pre-filter before the one-per-week rule
            continue
        r = money(val, give)
        rows.append({"rule": " AND ".join(combo), "discounts": r["discounts_sent"], "wasted_eur": r["wasted_eur"],
                     "left_without_nudge": r["left_without_nudge"]})
search = pd.DataFrame(rows)
matched = search[(search["discounts"] - target).abs() <= 0.15 * target].sort_values("wasted_eur").reset_index(drop=True)
print(f"Nudge@60% sends {target:,} discounts on validation; {len(search):,} rules in range, {len(matched)} within 15%")
display(matched.head(10).round(1))
print('the best 3 rules by validation € wasted per discount-matched count; also, the rule closest in discounts:')
best_rule = matched.loc[0, "rule"]
print("best segment rule on validation:", best_rule)

Nudge@60% sends 12,195 discounts on validation; 167 rules in range, 15 within 15%


,rule,discounts,wasted_eur,left_without_nudge
0,4+ clicks AND cart value <= €458,10788,49772.7,8932
1,cart value > €37 AND cart value <= €458,11411,54500.7,8437
2,4+ clicks AND cart value > €37 AND cart value ...,11377,65902.0,8496
3,4+ clicks AND cart value <= €946,12656,69842.8,7410
4,returning visitor AND cart value <= €946,10872,71048.3,9181
5,1+ visit in last 14 d AND cart value <= €946,10872,71048.3,9181
6,returning visitor AND 1+ visit in last 14 d AN...,10872,71048.3,9181
7,cart value > €89 AND cart value <= €946,11331,73570.4,8569
8,cart value > €37 AND cart value <= €946,13655,79583.6,6600
9,4+ clicks AND cart value > €89,10606,90204.3,9241


the best 3 rules by validation € wasted per discount-matched count; also, the rule closest in discounts:
best segment rule on validation: 4+ clicks AND cart value <= €458


In [5]:
def parse_atoms(d, names):
    cvv = d["cart_value"]
    thr = {q: cv.quantile(q) for q in (0.1, 0.25, 0.5, 0.75, 0.9)}      # thresholds come from VALIDATION
    table = {"new visitor": d["returning"].eq(False), "returning visitor": d["returning"],
             "bought in last 14 d": d["bought_14d"], "did not buy in last 14 d": ~d["bought_14d"],
             "no visit in last 14 d": d["visits_14d"].eq(0), "1+ visit in last 14 d": d["visits_14d"].ge(1),
             "3+ visits in last 14 d": d["visits_14d"].ge(3),
             "1 to 3 clicks": d["n_clicks"].le(3), "4+ clicks": d["n_clicks"].ge(4), "8+ clicks": d["n_clicks"].ge(8), "15+ clicks": d["n_clicks"].ge(15), "2 or fewer clicks": d["n_clicks"].le(2)}
    for q, t in thr.items():
        table[f"cart value <= €{t:,.0f}"] = cvv.le(t)
        table[f"cart value > €{t:,.0f}"] = cvv.gt(t)
    return np.logical_and.reduce([table[n].to_numpy() for n in names])
segment = lambda d: parse_atoms(d, best_rule.split(" AND "))   # noqa: E731
assert (segment(val) == np.logical_and.reduce([atoms[c] for c in best_rule.split(" AND ")])).all()
matched.head(20).round(2).to_csv(OUT / "smart_workflows_segment_search.csv", index=False)

## 4. The comparison on test
All numbers on the **test** cart abandoners. `€ wasted` = 10% of what shoppers who got the discount and came back anyway really bought. `left without a nudge` = cart abandoners who got **no discount** and did not come back (escalation: only those removed by the one-per-week rule, and every shopper still got a reminder; see the catches below). `reminders` = plain reminders sent (escalation only).

In [6]:
masks = policy_masks(test)
masks[f"Segment rule: {best_rule}"] = segment(test)
order = list(masks)

def numbers(d):
    out = {}
    msk = policy_masks(d)
    msk[f"Segment rule: {best_rule}"] = segment(d)
    for p, g in msk.items():
        r = money(d, g)
        out[(p, "discounts")] = r["discounts_sent"]
        out[(p, "wasted_eur")] = r["wasted_eur"]
        out[(p, "left_without_nudge")] = r["left_without_nudge"]
        out[(p, "wasted_per_discount")] = r["wasted_eur"] / max(r["discounts_sent"], 1)
    return out

point = numbers(test)
ranges = bootstrap(test, lambda d: {f"{p}|{k}": v for (p, k), v in numbers(d).items()}, n=BOOT_N, seed=SEED)
rule_w = point[(order[0], "wasted_eur")]
rows = []
for p in order:
    r = {"policy": p, "cart_abandoners": len(test),
         "reminders": len(test) if p.startswith("Escalation") else 0,
         "discounts": point[(p, "discounts")],
         "wasted_eur": point[(p, "wasted_eur")], "wasted_eur_lo": ranges[f"{p}|wasted_eur"][0], "wasted_eur_hi": ranges[f"{p}|wasted_eur"][1],
         "wasted_vs_rule": point[(p, "wasted_eur")] / rule_w - 1,
         "wasted_eur_per_discount": point[(p, "wasted_per_discount")],
         "wasted_per_discount_lo": ranges[f"{p}|wasted_per_discount"][0], "wasted_per_discount_hi": ranges[f"{p}|wasted_per_discount"][1],
         "left_without_nudge": point[(p, "left_without_nudge")],
         "left_lo": ranges[f"{p}|left_without_nudge"][0], "left_hi": ranges[f"{p}|left_without_nudge"][1]}
    rows.append(r)
table = pd.DataFrame(rows).set_index("policy")
display(table.round(3))
table.round(4).to_csv(OUT / "smart_workflows.csv")

,cart_abandoners,reminders,discounts,wasted_eur,wasted_eur_lo,wasted_eur_hi,wasted_vs_rule,wasted_eur_per_discount,wasted_per_discount_lo,wasted_per_discount_hi,left_without_nudge,left_lo,left_hi
policy,,,,,,,,,,,,,
Industry rule (everyone gets 10%),22384,0,16467,119728.780,110337.730,128151.870,0.000,7.271,6.711,7.774,4080,3914.575,4244.100
"Escalation: reminder, 10% after 24 h if not bought",22384,22384,15443,72523.027,66600.364,78647.742,-0.394,4.696,4.314,5.102,3967,3793.850,4128.125
"Escalation: reminder, 10% after 48 h if not bought",22384,22384,14763,42317.035,38477.238,47772.314,-0.647,2.866,2.611,3.221,3890,3728.625,4043.250
Nudge@60%,22384,0,11987,68785.309,63207.690,73911.665,-0.425,5.738,5.283,6.154,7315,7110.000,7532.125
Nudge@60% + escalation 24 h,22384,0,11313,41138.963,37321.328,44864.369,-0.656,3.636,3.296,3.956,7301,7100.000,7520.025
Segment rule: 4+ clicks AND cart value <= €458,22384,0,10719,51584.162,48214.090,55106.485,-0.569,4.812,4.493,5.147,8838,8616.925,9026.450


## 5. What this means, and the catches
Read the table first; the text below is filled in by the next cell from the numbers, so nothing is typed by hand.

In [7]:
t = table
n60, e24, e48, seg, ne = (t.loc["Nudge@60%"], t.loc["Escalation: reminder, 10% after 24 h if not bought"],
                          t.loc["Escalation: reminder, 10% after 48 h if not bought"], t.loc[f"Segment rule: {best_rule}"],
                          t.loc["Nudge@60% + escalation 24 h"])
lines = [
 f"Rule (everyone): {t.iloc[0].discounts:,.0f} discounts, EUR {rule_w:,.0f} wasted, {t.iloc[0].left_without_nudge:,.0f} non-returning cart abandoners without a nudge.",
 f"Escalation 24 h: {e24.discounts:,.0f} discounts, EUR {e24.wasted_eur:,.0f} wasted ({e24.wasted_vs_rule:+.0%} vs rule); 48 h: {e48.discounts:,.0f} discounts, EUR {e48.wasted_eur:,.0f} ({e48.wasted_vs_rule:+.0%}).",
 f"Segment rule ({best_rule}): {seg.discounts:,.0f} discounts, EUR {seg.wasted_eur:,.0f} ({seg.wasted_vs_rule:+.0%}), {seg.left_without_nudge:,.0f} left without a nudge.",
 f"Nudge@60%: {n60.discounts:,.0f} discounts, EUR {n60.wasted_eur:,.0f} ({n60.wasted_vs_rule:+.0%}), {n60.left_without_nudge:,.0f} left without a nudge.",
 f"Nudge + escalation 24 h: {ne.discounts:,.0f} discounts, EUR {ne.wasted_eur:,.0f} ({ne.wasted_vs_rule:+.0%}), {ne.left_without_nudge:,.0f} left without a nudge.",
]
print("\n".join("- " + l for l in lines))
print("\nwasted EUR 95% ranges:", {p: f"{r.wasted_eur_lo:,.0f}-{r.wasted_eur_hi:,.0f}" for p, r in t.iterrows()})

- Rule (everyone): 16,467 discounts, EUR 119,729 wasted, 4,080 non-returning cart abandoners without a nudge.
- Escalation 24 h: 15,443 discounts, EUR 72,523 wasted (-39% vs rule); 48 h: 14,763 discounts, EUR 42,317 (-65%).
- Segment rule (4+ clicks AND cart value <= €458): 10,719 discounts, EUR 51,584 (-57%), 8,838 left without a nudge.
- Nudge@60%: 11,987 discounts, EUR 68,785 (-43%), 7,315 left without a nudge.
- Nudge + escalation 24 h: 11,313 discounts, EUR 41,139 (-66%), 7,301 left without a nudge.

wasted EUR 95% ranges: {'Industry rule (everyone gets 10%)': '110,338-128,152', 'Escalation: reminder, 10% after 24 h if not bought': '66,600-78,648', 'Escalation: reminder, 10% after 48 h if not bought': '38,477-47,772', 'Nudge@60%': '63,208-73,912', 'Nudge@60% + escalation 24 h': '37,321-44,864', 'Segment rule: 4+ clicks AND cart value <= €458': '48,214-55,106'}


**Catches to say openly**
1. **Escalation wastes less only by waiting.** It cannot tell who would come back; it waits and sees. A shopper who buys within D hours costs nothing. But the data cannot say whether the delay itself loses sales: a discount that arrives 24 or 48 hours later may reach a person who has already gone elsewhere. Our data has no messages, so a delay cannot be measured here; only an A/B test can say.
2. **Escalation never skips a non-returner on purpose.** Every shopper who does not come back gets the late discount, so its "left without a nudge" is only the people removed by the one-per-week rule (they still got the reminder). That is the "margin to people who were only browsing" problem again: look at the discounts column and the EUR per discount, not at the last column alone. Escalation also needs the shopper to be reachable after 24 or 48 hours.
3. The one-per-week rule is applied on the decision time for escalation, a small simplification.
4. **A cart-value segment rule can beat Nudge on EUR wasted.** The 10% is worth 10% of the order, so skipping big carts saves the most euros, whatever the shopper's intent. Such a rule saves euros but it also skips 8,838 non-returning cart abandoners (Nudge: 7,315) and it ignores behaviour; and Nudge can use the same lever (see the combined row). Chosen on validation, reported on test; the 95% ranges tell how close the rows are.
5. Prediction is not causation: all policies are scored on who comes back without help, not on sales the discount causes.